In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd

train_df = pd.read_parquet("../data/processed/casp9_features.parquet")
val_df = pd.read_parquet("../data/processed/validation_features.parquet")
test_df = pd.read_parquet("../data/processed/testing_features.parquet")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (3337092, 51)
Validation shape: (47602, 51)
Test shape: (24466, 51)


The used input vector is those 41 features we extracted :- 21 Evo columns and 20 Amino acid Residue encodings 

In [3]:
feature_cols = [c for c in train_df.columns if c.startswith("Evo") or c.startswith("AA_")]

print("Number of features:", len(feature_cols))
print(feature_cols[:10])

Number of features: 41
['Evo1', 'Evo2', 'Evo3', 'Evo4', 'Evo5', 'Evo6', 'Evo7', 'Evo8', 'Evo9', 'Evo10']


Here the target feature is CentroidDist

In [4]:


X_train = train_df[feature_cols]
y_train = train_df["CentroidDist"]

X_test = test_df[feature_cols]
y_test = test_df["CentroidDist"]

Feature scaling 

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Pipeline 1 ( Linear Regression )

In [7]:
print("Training Linear Regression")

lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

preds_lr = lr.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, preds_lr))
mae = mean_absolute_error(y_test, preds_lr)
r2 = r2_score(y_test, preds_lr)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Training Linear Regression
RMSE: 910.5489579222362
MAE: 646.0605545553425
R2: 0.03322808591369797


Pipleline 1 ( SVM with sampling )

In [ ]:
print("Training SVR")

sample_size = 200000

sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)

X_svr = X_train_scaled[sample_idx]
y_svr = y_train.iloc[sample_idx]

svr = SVR(kernel="rbf")

svr.fit(X_svr, y_svr)

preds_svr = svr.predict(X_test_scaled[:100000])

rmse = np.sqrt(mean_squared_error(y_test[:100000], preds_svr))
mae = mean_absolute_error(y_test[:100000], preds_svr)
r2 = r2_score(y_test[:100000], preds_svr)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Training SVR
RMSE: 902.5429709117792
MAE: 605.7903000366231
R2: 0.05015399982014612


: 